<a href="https://colab.research.google.com/github/draginverse/dragin-healthcare/blob/feature%2Fg-retriever/scripts/retrieval/similarity_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pymilvus

In [2]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import os
import pandas as pd
from pathlib import Path

from pymilvus import MilvusClient, Collection, FieldSchema, CollectionSchema, DataType

import time
import random

# triplets

In [11]:
# Toy graph data
toy_graphs = [
    [
        ("asthma", "caused_by", "allergens"),
        ("inhaler", "treats", "asthma"),
        ("asthma", "symptom", "shortness of breath")
    ],
    [
        ("copd", "risk_factor", "smoking"),
        ("oxygen therapy", "treats", "copd"),
        ("copd", "symptom", "chronic cough")
    ],
    [
        ("bronchitis", "caused_by", "virus"),
        ("bronchitis", "symptom", "chest discomfort"),
        ("rest", "helps_with", "bronchitis")
    ],
    [
        ("pneumonia", "caused_by", "bacteria"),
        ("antibiotics", "treats", "pneumonia"),
        ("pneumonia", "symptom", "fever")
    ],
    [
        ("covid-19", "affects", "lungs"),
        ("vaccine", "prevents", "covid-19"),
        ("covid-19", "symptom", "loss of smell")
    ]
]


In [12]:
# --- Graph Text Representation ---
def graph_to_text(graph):
    return " ".join([f"{s} {r} {o}." for s, r, o in graph])

In [13]:
# --- Graph Embedder ---
class GraphEmbedder:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)

    def embed_graphs(self, graphs, text_fn=graph_to_text):
        graph_texts = [text_fn(g) for g in graphs]
        embeddings = self.model.encode(graph_texts, convert_to_tensor=True)
        return graph_texts, embeddings

    def embed_query(self, query):
        return self.model.encode(query, convert_to_tensor=True)


In [14]:
# --- Similarity-Based Retriever ---
class GraphRetriever:
    def __init__(self, graphs, graph_texts, graph_embeddings):
        self.graphs = graphs
        self.graph_texts = graph_texts
        self.graph_embeddings = graph_embeddings

    def retrieve(self, query_embedding, top_k=1):
        scores = util.cos_sim(query_embedding, self.graph_embeddings)[0]
        top_k = min(top_k, len(self.graphs))
        top_results = scores.topk(k=top_k)
        top_indices = top_results.indices.tolist()
        top_scores = top_results.values.tolist()
        return [(self.graphs[i], self.graph_texts[i], top_scores[idx]) for idx, i in enumerate(top_indices)]

In [ ]:
if __name__ == "__main__":
    embedder = GraphEmbedder()
    graph_texts, graph_embeddings = embedder.embed_graphs(toy_graphs)

    retriever = GraphRetriever(toy_graphs, graph_texts, graph_embeddings)

    query = "i need a cure for asthma"
    query_embedding = embedder.embed_query(query)

    top_k_results = retriever.retrieve(query_embedding, top_k=3)

    print(f"\nQuery: {query}\n")

    for rank, (graph, text, score) in enumerate(top_k_results, 1):
        print(f"\nTop-{rank} Graph (Score: {score:.4f}):\n{text}")
        for triple in graph:
            print(triple)


Query: i need a cure for asthma


Top-1 Graph (Score: 0.5395):
asthma caused_by allergens. inhaler treats asthma. asthma symptom shortness of breath.
('asthma', 'caused_by', 'allergens')
('inhaler', 'treats', 'asthma')
('asthma', 'symptom', 'shortness of breath')

Top-2 Graph (Score: 0.3641):
copd risk_factor smoking. oxygen therapy treats copd. copd symptom chronic cough.
('copd', 'risk_factor', 'smoking')
('oxygen therapy', 'treats', 'copd')
('copd', 'symptom', 'chronic cough')

Top-3 Graph (Score: 0.3439):
covid-19 affects lungs. vaccine prevents covid-19. covid-19 symptom loss of smell.
('covid-19', 'affects', 'lungs')
('vaccine', 'prevents', 'covid-19')
('covid-19', 'symptom', 'loss of smell')


# csv

In [ ]:
# --- Module: Graph Loader from Folder Structure ---
class CSVGraphLoader:
    def __init__(self, nodes_folder, edges_folder):
        self.nodes_folder = Path(nodes_folder)
        self.edges_folder = Path(edges_folder)
        if not self.nodes_folder.exists() or not self.edges_folder.exists():
            raise ValueError("Folder structure not found.")

    def load_graph(self, graph_id):
        # Load nodes
        node_path = self.nodes_folder / f"{graph_id}.csv"
        edge_path = self.edges_folder / f"{graph_id}.csv"

        node_df = pd.read_csv(node_path)
        edge_df = pd.read_csv(edge_path)

        # Build a mapping from node id to content
        node_map = dict(zip(node_df['node_id'], node_df['node_attr']))

        # Build graph as list of (source_content, edge_label, destination_content)
        triples = []
        for _, row in edge_df.iterrows():
            src = node_map.get(row['src'], f"NODE_{row['src']}")
            dst = node_map.get(row['dst'], f"NODE_{row['dst']}")
            triples.append((src, row['edge_attr'], dst))

        return triples

    def load_all_graphs(self):
        graphs = []
        files = list(self.nodes_folder.glob("*.csv"))

        graph_ids = []
        for f in files:
            try:
                id_val = int(f.stem)
                graph_ids.append(id_val)
            except ValueError:
                print(f"Skipping non-integer filename: {f.name}")

        graph_ids = sorted(graph_ids)

        print(f"Found graph IDs: {graph_ids}")

        for graph_id in graph_ids:
            graph = self.load_graph(graph_id)
            graphs.append(graph)
        return graphs

In [ ]:
if __name__ == "__main__":
    # Load CSV-based graphs
    loader = CSVGraphLoader("/content/nodes", "/content/edges")
    csv_graphs = loader.load_all_graphs()

    # Embed and retrieve just like before
    embedder = GraphEmbedder()
    graph_texts, graph_embeddings = embedder.embed_graphs(csv_graphs)
    retriever = GraphRetriever(csv_graphs, graph_texts, graph_embeddings)

    query = "tropical weather"
    query_embedding = embedder.embed_query(query)
    top_k_results = retriever.retrieve(query_embedding, top_k=3)

    for rank, (graph, text, score) in enumerate(top_k_results, 1):
        print(f"\nTop-{rank} Graph (Score: {score:.4f}):")

        # Print only the first few triples
        preview = graph[:3]  # change to 2 if you only want 2
        for triple in preview:
            print(" ", triple)

        if len(graph) > len(preview):
            print(f"  ... and {len(graph) - len(preview)} more triples.")


Found graph IDs: [0, 1, 2, 3, 4, 5]

Top-1 Graph (Score: 0.4075):
  ('hurricane betsy', 'common.topic.notable_types', 'tropical cyclone')
  ('united states of america', 'meteorology.cyclone_affected_area.cyclones', 'hurricane isabel')
  ('bahamas', 'location.statistical_region.co2_emissions_per_capita', 'g.12460kjms')
  ... and 5731 more triples.

Top-2 Graph (Score: 0.2966):
  ('acklins', 'location.administrative_division.first_level_division_of', 'bahamas')
  ('geographical feature', 'freebase.type_profile.strict_included_types', 'location')
  ('hurricane edith', 'meteorology.tropical_cyclone.affected_areas', 'bahamas')
  ... and 2171 more triples.

Top-3 Graph (Score: 0.0819):
  ('john noble', 'film.actor.film', 'm.0h1ldcx')
  ('m.02_1x1x', 'base.gender.personal_gender_identity.gender_identity', 'male')
  ('m.0b4d5rz', 'award.award_nomination.award_nominee', 'andy serkis')
  ... and 1487 more triples.


# milvus db

In [22]:
CLUSTER_ENDPOINT = "https://in03-7a5f9d2a1aa84ef.serverless.gcp-us-west1.cloud.zilliz.com"
API_KEY = "a73c79fb1924d05aeb410abc0d5669293cc33be37a123953be640725aa42198ef5c1e499cc07f231977c742ad6e6977c6eddec05"

In [23]:
milvus_client = MilvusClient(uri=CLUSTER_ENDPOINT, token=API_KEY)

In [24]:
# Create and Load Milvus Collection
collection_name = "test_graphs"
if not milvus_client.has_collection(collection_name):
    # Preparing schema
    dim = 384
    schema = milvus_client.create_schema()
    schema.add_field("id", DataType.INT64, is_primary=True, auto_id=True)
    schema.add_field("embedding", DataType.FLOAT_VECTOR, dim=dim)
    schema.add_field("graph", DataType.VARCHAR, max_length=9999)

    # Preparing index parameters
    index_params = milvus_client.prepare_index_params()
    index_params.add_index("embedding", index_type="IVF_FLAT", metric_type="COSINE", index_params={"nlist": 64})

    # Create collection with the above schema and index parameters, and then load automatically
    milvus_client.create_collection(collection_name, dimension=dim, schema=schema, index_params=index_params)
else:
    print(f"Collection '{collection_name}' already exists.")

Collection 'test_graphs' already exists.


In [25]:
embedder = GraphEmbedder()
graph_texts, graph_embeddings = embedder.embed_graphs(toy_graphs)

In [16]:
# Insert the Data
data = [{"embedding": embedding, "graph": chunk} for embedding, chunk in zip(graph_embeddings, graph_texts)]

milvus_client.insert(collection_name=collection_name, data=data)

{'insert_count': 5, 'ids': [458038124103965193, 458038124103965194, 458038124103965195, 458038124103965196, 458038124103965197], 'cost': 3}

In [26]:
# Defining Semantic search function
def semantic_search(query, top_k=2):
    query_vec = SentenceTransformer('all-MiniLM-L6-v2').encode([query]).tolist()
    results = milvus_client.search(
        collection_name=collection_name,
        data=query_vec,
        limit=top_k,
        output_fields=["graph"],
        search_params={"metric_type": "COSINE", "params": {"nprobe": 10}},
    )
    return results

# Executing search
search_query = "i need to cure my cough"
nb_closest = 5
search_res = semantic_search(search_query, nb_closest)

for hit in search_res[0]:
    id = hit["id"]
    similarity = hit["distance"] # not to confuse distance for dissimilarity here
    text = hit["entity"]["graph"]
    print(f"Score: {round(similarity, 4)} \t| Text: {text} \t| ID: {id}")

Score: 0.341 	| Text: covid-19 affects lungs. vaccine prevents covid-19. covid-19 symptom loss of smell. 	| ID: 458038124103965197
Score: 0.3133 	| Text: copd risk_factor smoking. oxygen therapy treats copd. copd symptom chronic cough. 	| ID: 458038124103965194
Score: 0.2833 	| Text: bronchitis caused_by virus. bronchitis symptom chest discomfort. rest helps_with bronchitis. 	| ID: 458038124103965195
Score: 0.2528 	| Text: pneumonia caused_by bacteria. antibiotics treats pneumonia. pneumonia symptom fever. 	| ID: 458038124103965196
Score: 0.1913 	| Text: asthma caused_by allergens. inhaler treats asthma. asthma symptom shortness of breath. 	| ID: 458038124103965193
